# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Description: Evaluate the following term dispersion score/keyword extraction methods on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)
- KeyBERT
- KeyLLM

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords.

This version of the code includes singletons from the analysis.

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
lex_size = len(genia_lexical_units_and_sems) # Number of lexical units in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(lex_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [4]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [5]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [6]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/2-tables/singletons-included-analysis/../../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [7]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, genia_lexical_units_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753


Term dispersion scores:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


In [8]:
# Integrate KeyBERT scores

# Load KeyBERT results 
keybert_keywords_path = '../keybert-scores.tsv'

with open(keybert_keywords_path, "r") as c:
  keybert_scores_df = pd.read_csv(c, sep='\t')

keybert_scores_df['sem'] = keybert_scores_df['sem'].fillna(str())

# Add KeyBERT scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keybert_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyBERT'] = term_scores_aug_df['KeyBERT'].fillna(0)

# Check for duplicate terms
all_duplicates = keybert_scores_df[keybert_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keybert_scores_df = keybert_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyBERT
3132,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,0.0010
16199,minus_clone_lex,G#cell_line,0.0005
18608,octamer_motif_lex,G#DNA_domain_or_region,0.0035


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.0010,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.0005,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.0035,0.808753


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,-0.000251


In [9]:
# Integrate KeyLLM scores

# Load KeyLLM results 
keyllm_keywords_path = '../keyllm-scores.tsv'

with open(keyllm_keywords_path, "r") as c:
  keyllm_scores_df = pd.read_csv(c, sep='\t')

keyllm_scores_df['sem'] = keyllm_scores_df['sem'].fillna(str())

# Check for duplicate terms
all_duplicates = keyllm_scores_df[keyllm_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keyllm_scores_df = keyllm_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Print to console
print("KeyLLM scores:")
display(keyllm_scores_df)

# Add KeyLLM scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keyllm_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyLLM'] = term_scores_aug_df['KeyLLM'].fillna(0)

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'KeyLLM', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyLLM
136,20-epi_analogue_lex,G#other_organic_compound,0.0005
137,20-Epi_analogue_lex,G#other_organic_compound,0.0005
177,25-dihydroxyvitamin_d3,,0.0015
192,25_dihydroxyvitamin_d3,,0.0005
413,9-cis_RA_lex,G#other_organic_compound,0.0005
...,...,...,...
18825,pRb_lex,G#protein_molecule,0.0010
18826,pRB_lex,G#protein_family_or_group,0.0010
22321,v-abl_lex,G#DNA_domain_or_region,0.0005
22322,v-Abl_lex,G#protein_molecule,0.0005


KeyLLM scores:


,term,sem,KeyLLM
0,(3H)_dexamethasone_lex,G#lipid,0.0005
1,(Ca2+)i_lex,G#inorganic,0.0005
2,(Ca2+)i_requirement_for_lex,G#other_name,0.0005
3,-120_region_lex,G#DNA_domain_or_region,0.0005
4,-130_AP-1-like_site_lex,G#DNA_domain_or_region,0.0005
...,...,...,...
33986,CD4_negative_T_cell_line_lex,G#cell_line,0.0000
33987,gp_160-induced_nuclear_extract_lex,G#cell_component,0.0000
33988,gp160-induced_AP-1_complex_lex,G#protein_complex,0.0000
33989,protein_synthesis-independent_lex,G#other_name,0.0000


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,0.0,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,0.0,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,0.0,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40800,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40801,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,0.0,-0.000251
40802,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,0.0,-0.000251


In [10]:
# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t', index=False)

## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [11]:
# Desginated ordering for the high-level semantic classes
high_level_semantic_class_ord = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
lex_unit_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['term'].nunique().reindex(high_level_semantic_class_ord).to_list()

# Count number of annotations associated with each high-level semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Count number of singletons associated with each high-level semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Initialize GENIA summary statistics data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': high_level_semantic_class_ord,
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print GENIA summary statistics to console
display(genia_summary_stats_df)

,Semantic class,Sub-class,Unique terms,Annotations,Singletons
0,amino_acid,15,10155,42478,6571
1,nucleotide,12,5574,11619,4115
2,multi_cell,5,1444,5247,961
3,cell,4,4051,11626,2956
4,other,1,10560,19999,8071


## Terminology Extraction Task Experiment

Here we reproduce the results tables for the GENIA analysis.

In [13]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'class', 'ni', 'bi'])

# Print to console
display(term_scores_df)

# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-minimal.tsv', sep='\t', index=False)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,'aged'_lymphocyte_lex,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,0.0,-0.000251
1,'converted'_TCEd_motif_lex,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,0.0,-0.000251
2,'latency_I'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
3,'latency_II'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
4,'master_regulator_lex,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,0.0,-0.000251
...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40800,zymogen_plasma_factors_VII_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40801,zymography_lex,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,0.0,-0.000251
40802,zymosan-treated_cell_lex,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,0.0,-0.000251


Define various functions used in the analysis.

In [14]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  #print(measures)
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [15]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 100 # To test, set to 5

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                                                                | 0/100 [00:00<?, ?it/s]

r = 0


  1%|█▎                                                                                                                                   | 1/100 [03:19<5:29:55, 199.95s/it]

r = 1


  2%|██▋                                                                                                                                  | 2/100 [06:38<5:24:49, 198.88s/it]

...

 98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 98/100 [5:20:26<06:29, 194.85s/it]

r = 98


 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 99/100 [5:23:40<03:14, 194.76s/it]

r = 99


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [5:26:55<00:00, 196.15s/it]


Save evaluation metrics as Pkl files.

In [16]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [17]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-a4', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-a4/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-a4/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-a4/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-a4/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-a4/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-a4/other-fk-means.csv', index=False)

In [18]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0005,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006
50,0.0027,0.0027,0.0030,0.0030,0.0031,0.0024,0.0031,0.0031,0.0031
100,0.0054,0.0054,0.0060,0.0061,0.0061,0.0049,0.0063,0.0063,0.0063
500,0.0267,0.0263,0.0295,0.0305,0.0302,0.0256,0.0310,0.0310,0.0307
1000,0.0525,0.0519,0.0581,0.0599,0.0588,0.0521,0.0610,0.0610,0.0601
5000,0.2339,0.2312,0.2488,0.2523,0.2439,0.2379,0.2718,0.2718,0.2532


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0011,0.0020,0.0016,0.0003,0.0012,0.0012,0.0020
50,0.0026,0.0025,0.0052,0.0074,0.0069,0.0021,0.0040,0.0055,0.0078
100,0.0051,0.0049,0.0104,0.0160,0.0145,0.0026,0.0082,0.0105,0.0162
500,0.0246,0.0231,0.0504,0.0645,0.0584,0.0206,0.0438,0.0446,0.0649
1000,0.0476,0.0441,0.0967,0.1155,0.1058,0.0453,0.0810,0.0824,0.1152
5000,0.1748,0.1631,0.2831,0.2826,0.2711,0.1848,0.2410,0.2591,0.2833


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0006,0.0006,0.0000,0.0004,0.0000,0.0004,0.0000,0.0000
50,0.0028,0.0028,0.0028,0.0018,0.0021,0.0021,0.0014,0.0007,0.0018
100,0.0056,0.0054,0.0056,0.0035,0.0035,0.0051,0.0025,0.0032,0.0039
500,0.0257,0.0257,0.0253,0.0233,0.0240,0.0224,0.0146,0.0192,0.0235
1000,0.0475,0.0472,0.0465,0.0409,0.0425,0.0426,0.0339,0.0371,0.0416
5000,0.1482,0.1458,0.1446,0.1469,0.1402,0.1444,0.1333,0.1367,0.1473


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0004,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0025,0.0023,0.0030,0.0027,0.0027,0.0054,0.0067,0.0027,0.0027
100,0.0049,0.0044,0.0056,0.0026,0.0026,0.0117,0.0117,0.0065,0.0026
500,0.0188,0.0179,0.0209,0.0172,0.0165,0.0289,0.0280,0.0196,0.0174
1000,0.0295,0.0294,0.0329,0.0300,0.0344,0.0434,0.0462,0.0323,0.0293
5000,0.0567,0.0558,0.0679,0.0680,0.0640,0.0726,0.0761,0.0750,0.0685


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0006,0.0004,0.0000,0.0000,0.0004,0.0005,0.0010,0.0000
50,0.0025,0.0028,0.0019,0.0002,0.0000,0.0038,0.0038,0.0049,0.0002
100,0.0050,0.0053,0.0039,0.0005,0.0010,0.0058,0.0082,0.0072,0.0005
500,0.0242,0.0241,0.0181,0.0085,0.0132,0.0198,0.0341,0.0353,0.0088
1000,0.0433,0.0441,0.0325,0.0236,0.0273,0.0364,0.0550,0.0564,0.0237
5000,0.1220,0.1225,0.1054,0.1106,0.1091,0.1046,0.1369,0.1363,0.1112


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0005,0.0003,0.0000,0.0002,0.0012,0.0004,0.0004,0.0000
50,0.0028,0.0029,0.0013,0.0006,0.0013,0.0019,0.0025,0.0015,0.0006
100,0.0056,0.0058,0.0026,0.0006,0.0018,0.0055,0.0047,0.0032,0.0006
500,0.0268,0.0274,0.0125,0.0074,0.0103,0.0292,0.0212,0.0189,0.0075
1000,0.0508,0.0524,0.0240,0.0184,0.0213,0.0546,0.0417,0.0409,0.0190
5000,0.1875,0.1944,0.1247,0.1288,0.1273,0.1935,0.2062,0.1871,0.1295
